In [14]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import nltk
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [3]:
nltk.download('treebank')

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

In [4]:
from nltk.corpus import treebank

sentences = []
tags = []

for sent in treebank.tagged_sents():
    words = []
    pos_tags = []
    
    for word, tag in sent:
        words.append(word.lower())
        pos_tags.append(tag)
    
    sentences.append(words)
    tags.append(pos_tags)

print(sentences[0])
print(tags[0])

['pierre', 'vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'nov.', '29', '.']
['NNP', 'NNP', ',', 'CD', 'NNS', 'JJ', ',', 'MD', 'VB', 'DT', 'NN', 'IN', 'DT', 'JJ', 'NN', 'NNP', 'CD', '.']


In [5]:
train_sentences, test_sentences, train_tags, test_tags = train_test_split(
    sentences, tags, test_size=0.2, random_state=42
)

In [ ]:
from collections import Counter

def build_vocab(sentences, min_freq=1):
    counter = Counter()
    for sent in sentences:
        counter.update(sent)

    word2idx = {'<PAD>': 0, '<UNK>': 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            word2idx[word] = len(word2idx)

    return word2idx

word2idx = build_vocab(train_sentences)


tag_set = set(tag for seq in train_tags for tag in seq)
tag2idx = {tag: i for i, tag in enumerate(tag_set)}
idx2tag = {i: tag for tag, i in tag2idx.items()}

In [ ]:
def encode(sentences, tags, word2idx, tag2idx, max_len=50):
    X, y = [], []

    for sent, tag_seq in zip(sentences, tags):
        word_ids = [word2idx.get(w, word2idx['<UNK>']) for w in sent]
        tag_ids = [tag2idx[t] for t in tag_seq]

        word_ids = word_ids[:max_len]
        tag_ids = tag_ids[:max_len]

        pad_len = max_len - len(word_ids)
        word_ids += [word2idx['<PAD>']] * pad_len
        tag_ids += [-100] * pad_len   
        X.append(word_ids)
        y.append(tag_ids)

    return torch.tensor(X), torch.tensor(y)

max_len = 50

X_train, y_train = encode(train_sentences, train_tags, word2idx, tag2idx, max_len)
X_test, y_test = encode(test_sentences, test_tags, word2idx, tag2idx, max_len)

In [ ]:
class RNNTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, tagset_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)  
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.dropout(out)   
        out = self.fc(out)
        return out

In [ ]:
vocab_size = len(word2idx)
tagset_size = len(tag2idx)

model = RNNTagger(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128,
    tagset_size=tagset_size
)

criterion = nn.CrossEntropyLoss(ignore_index=-100) 

optimizer = optim.Adam(model.parameters(), lr=0.001)  

In [ ]:
def train(model, X, y, epochs=20, batch_size=32):
    model.train()

    for epoch in range(epochs):
        total_loss = 0

       
        perm = torch.randperm(len(X))
        X_shuffled = X[perm]
        y_shuffled = y[perm]

        for i in range(0, len(X), batch_size):
            xb = X_shuffled[i:i+batch_size]
            yb = y_shuffled[i:i+batch_size]

            optimizer.zero_grad()

            outputs = model(xb)

            loss = criterion(
                outputs.view(-1, outputs.shape[-1]),
                yb.view(-1)
            )

            loss.backward()

            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [34]:
train(model, X_train, y_train, epochs=30)

Epoch 1, Loss: 201.6695
Epoch 2, Loss: 111.1645
Epoch 3, Loss: 85.5823
Epoch 4, Loss: 70.0485
Epoch 5, Loss: 59.1673
Epoch 6, Loss: 50.5848
Epoch 7, Loss: 43.6312
Epoch 8, Loss: 37.9309
Epoch 9, Loss: 33.4237
Epoch 10, Loss: 29.2408
Epoch 11, Loss: 25.7213
Epoch 12, Loss: 22.9349
Epoch 13, Loss: 20.2549
Epoch 14, Loss: 18.1496
Epoch 15, Loss: 15.9490
Epoch 16, Loss: 14.3289
Epoch 17, Loss: 12.6547
Epoch 18, Loss: 11.5015
Epoch 19, Loss: 10.1993
Epoch 20, Loss: 9.2515
Epoch 21, Loss: 8.2838
Epoch 22, Loss: 7.4368
Epoch 23, Loss: 6.8675
Epoch 24, Loss: 6.3650
Epoch 25, Loss: 5.7803
Epoch 26, Loss: 5.2179
Epoch 27, Loss: 4.7528
Epoch 28, Loss: 4.3645
Epoch 29, Loss: 4.1090
Epoch 30, Loss: 3.7041


In [ ]:
def evaluate(model, X, y):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        outputs = model(X)
        preds = torch.argmax(outputs, dim=-1)

        for pred_seq, label_seq in zip(preds, y):
            for p, l in zip(pred_seq, label_seq):
                if l != -100:  
                    all_preds.append(p.item())
                    all_labels.append(l.item())

    acc = sum([p == l for p, l in zip(all_preds, all_labels)]) / len(all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

In [40]:
evaluate(model, X_test, y_test)

Accuracy: 0.9005
F1 Score: 0.9018


In [41]:
def predict(sentence):
    model.eval()

    words = sentence.lower().split()
    word_ids = [word2idx.get(w, word2idx['<UNK>']) for w in words]

    padded = word_ids + [0]*(max_len - len(word_ids))
    tensor = torch.tensor([padded])

    with torch.no_grad():
        outputs = model(tensor)
        preds = torch.argmax(outputs, dim=-1)[0]

    tags = [idx2tag[p.item()] for p in preds[:len(words)]]

    return list(zip(words, tags))


print(predict("the dog is running"))

[('the', 'DT'), ('dog', 'NNS'), ('is', 'VBZ'), ('running', 'VBG')]


In [38]:
print(predict("she is playing football"))
print(predict("they went to school"))
print(predict("i love machine learning"))

[('she', 'PRP'), ('is', 'VBZ'), ('playing', 'VBG'), ('football', 'NN')]
[('they', 'PRP'), ('went', 'VBD'), ('to', 'TO'), ('school', 'NN')]
[('i', 'PRP'), ('love', 'VBP'), ('machine', 'NN'), ('learning', 'NNP')]
